# NB_patient_360
Builds **`<catalog>.gold.patient_360`** from the silver layer, per STTM *Silver to Gold (CareSync Health Network) v1.0*.

* **Grain:** one row per patient
* **Driving table:** `<catalog>.silver.patients`
* **Joins (LEFT):** `silver.hospitals` on `registered_hospital_id = hospital_id`; `silver.insurance_providers` on `insurance_provider_id`; aggregated `silver.lab_results` grouped by `patient_id`
* **Filters:** none, all patients included
* **Load type:** FULL, `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
import re

dbutils.widgets.text("catalog", "adb_caresync_apollo", "Catalog")
catalog = dbutils.widgets.get("catalog").strip()

# Guard against injection: catalog must be a plain identifier
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError(f"Invalid catalog name: {catalog!r}")

silver = f"`{catalog}`.silver"
gold = f"`{catalog}`.gold"
target_table = f"{gold}.patient_360"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")
print(f"Target: {target_table}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per patient with hospital, insurance and lab-result summary. Source: STTM silver_to_gold v1.0'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count,
    MAX(test_date)                        AS last_test_date
  FROM {silver}.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                  AS patient_id,
  CAST(p.first_name AS STRING)                               AS first_name,
  CAST(p.last_name AS STRING)                                AS last_name,
  CAST(p.date_of_birth AS DATE)                              AS date_of_birth,
  CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  CAST(p.gender AS STRING)                                   AS gender,
  CAST(p.city AS STRING)                                     AS city,
  CAST(p.state AS STRING)                                    AS state,
  CAST(p.registered_hospital_id AS INT)                      AS hospital_id,
  CAST(h.hospital_name AS STRING)                            AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                       AS insurance_provider_id,
  CAST(ip.provider_name AS STRING)                           AS insurance_provider_name,
  CAST(COALESCE(la.total_tests, 0) AS INT)                   AS total_tests,
  CAST(COALESCE(la.abnormal_test_count, 0) AS INT)           AS abnormal_test_count,
  CAST(COALESCE(la.critical_test_count, 0) AS INT)           AS critical_test_count,
  CAST(la.last_test_date AS DATE)                            AS last_test_date
FROM {silver}.patients p
LEFT JOIN {silver}.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN {silver}.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg la
  ON p.patient_id = la.patient_id
""")

In [0]:
updated_df = spark.table(target_table)
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))